# Modelo: Most Popular

Baseline de ranking: recomienda los K restaurantes con más interacciones en train.

In [1]:
import os
import pandas as pd
import sys; sys.path.append('..')
from src.evaluation import temporal_train_test_split, evaluate_model

MODEL_NAME = 'mostpopular'
RESULTS_DIR = f'../results/{MODEL_NAME}'

## Definición del modelo

In [2]:
class MostPopularRecommender:
    def __init__(self):
        self._sorted_items = []

    def fit(self, train_reviews):
        counts = train_reviews['business_id'].value_counts()
        self._sorted_items = counts.index.tolist()
        return self

    def recommend(self, user_id, train_reviews, top_k=10):
        seen = set(train_reviews[train_reviews['user_id'] == user_id]['business_id'])
        return [i for i in self._sorted_items if i not in seen][:top_k]

## Datos

In [3]:
reviews = pd.read_csv('../data/processed/reviews.csv', parse_dates=['date'])
train_reviews, test_reviews = temporal_train_test_split(reviews, test_fraction=0.2)
print(f'Train: {len(train_reviews)} | Test: {len(test_reviews)}')

Train: 83256 reviews | Test: 17191 reviews
Train: 83256 | Test: 17191


## Entrenamiento y evaluación

In [4]:
model = MostPopularRecommender().fit(train_reviews)

metrics = evaluate_model(
    lambda uid, top_k: model.recommend(uid, train_reviews, top_k),
    test_reviews, train_reviews, k_values=[5, 10, 20]
)
print(metrics.round(4))

    precision  recall    ndcg
K                            
5      0.0089  0.0283  0.0211
10     0.0077  0.0506  0.0287
20     0.0068  0.0880  0.0391


## Guardar resultados

In [5]:
os.makedirs(RESULTS_DIR, exist_ok=True)
metrics.to_csv(f'{RESULTS_DIR}/metrics.csv')
print(f'Saved -> results/{MODEL_NAME}/metrics.csv')

Saved -> results/mostpopular/metrics.csv
